# PyTransformKit — Local Experimentation

Ce notebook est le premier point d'entrée d'expérimentation locale du framework.

Objectifs :

- construire un `Schema` logique ;
- définir un `Pipeline` engine-agnostic ;
- inspecter son `LogicalPlan` sans exécuter les données ;
- exécuter exactement le même Pipeline avec Pandas ;
- exécuter exactement le même Pipeline avec Polars ;
- comparer les résultats Pandas / Polars ;
- tester l'exécution Polars en mode lazy.

> Pré-requis local : `pip install -e ".[dev,pandas,polars]"`


## 1. Imports

Le Core reste indépendant des moteurs. Les adapters Pandas et Polars sont importés depuis `infrastructure` uniquement au moment de l'exécution physique.


In [ ]:
import pandas as pd
import polars as pl

from pytransformkit import (
    EngineRegistry,
    ExecutionContext,
    ExecutionMode,
    RunPipelineService,
)
from pytransformkit.domain.data.data_types import IntegerType, StringType
from pytransformkit.domain.data.field import Field
from pytransformkit.domain.data.schema import Schema
from pytransformkit.domain.pipelines import Pipeline, PipelinePlanner
from pytransformkit.functions import col, lower, trim
from pytransformkit.infrastructure.engines.pandas import (
    PandasAdapter,
    PandasDatasetHandle,
)
from pytransformkit.infrastructure.engines.polars import (
    PolarsAdapter,
    PolarsDatasetHandle,
)


## 2. Données d'exemple

On part d'un petit jeu de données client volontairement simple, avec quelques espaces et une valeur nulle dans l'email.


In [ ]:
records = [
    {
        "customer_id": 1,
        "email": " JOHN@EXAMPLE.COM ",
        "status": "ACTIVE",
    },
    {
        "customer_id": 2,
        "email": " ALICE@EXAMPLE.COM ",
        "status": "ACTIVE",
    },
    {
        "customer_id": 3,
        "email": None,
        "status": "INACTIVE",
    },
    {
        "customer_id": 4,
        "email": " BOB@EXAMPLE.COM ",
        "status": "ACTIVE",
    },
]

pandas_df = pd.DataFrame(records)
polars_df = pl.DataFrame(records)

pandas_df


## 3. Schema logique PyTransformKit

Le `Schema` appartient au Domain. Il ne dépend ni de Pandas ni de Polars.


In [ ]:
schema = Schema(
    fields=(
        Field(
            "customer_id",
            IntegerType(),
            nullable=False,
        ),
        Field(
            "email",
            StringType(),
            nullable=True,
        ),
        Field(
            "status",
            StringType(),
            nullable=False,
        ),
    )
)

schema


## 4. Définition d'un Pipeline engine-agnostic

Le même Pipeline sera ensuite envoyé aux adapters Pandas et Polars sans modifier sa définition.


In [ ]:
pipeline = (
    Pipeline.create(
        "customers",
        schema,
    )
    .filter(col("status") == "ACTIVE")
    .derive(
        "normalized_email",
        lower(trim(col("email"))),
    )
    .select(
        "customer_id",
        "normalized_email",
    )
)

pipeline


## 5. Inspection du LogicalPlan

Cette étape ne touche pas aux données physiques. Le framework valide le DAG et propage le Schema statiquement.


In [ ]:
planner = PipelinePlanner()
plan = planner.plan(pipeline)

print("Pipeline:", plan.pipeline_name)
print("Output schema:", plan.output_schema.names())
print("Node kinds:", [node.kind.value for node in plan.nodes])


## 6. Exécution avec Pandas

Le moteur est enregistré explicitement. Aucun fallback implicite n'est effectué par PyTransformKit.


In [ ]:
registry = EngineRegistry()
registry.register(PandasAdapter())

service = RunPipelineService(registry)

pandas_result = service.run(
    pipeline,
    PandasDatasetHandle(pandas_df),
    engine_id="pandas",
)

pandas_output = pandas_result.output_handle.dataframe
pandas_output


In [ ]:
print("Execution id:", pandas_result.execution_id)
print("Engine:", pandas_result.engine.id)
print("Output schema:", pandas_result.output_schema.names())


## 7. Exécution du même Pipeline avec Polars

Aucune modification n'est apportée au `Pipeline`, au `Schema`, aux expressions ou aux Transformations.


In [ ]:
registry.register(PolarsAdapter())

polars_result = service.run(
    pipeline,
    PolarsDatasetHandle(polars_df),
    engine_id="polars",
)

polars_output = polars_result.output_handle.frame
polars_output


## 8. Comparaison Pandas / Polars

On normalise les deux résultats sous forme de dictionnaires pour vérifier rapidement l'équivalence métier.


In [ ]:
pandas_records = pandas_output.to_dict(orient="records")
polars_records = polars_output.to_dicts()

print("Pandas:", pandas_records)
print("Polars:", polars_records)
print("Equivalent:", pandas_records == polars_records)


## 9. Polars LazyFrame

Polars annonce explicitement la capability `LAZY`. Le résultat reste un `LazyFrame` tant que l'utilisateur ne matérialise pas avec `collect()`.


In [ ]:
lazy_result = service.run(
    pipeline,
    PolarsDatasetHandle(polars_df.lazy()),
    engine_id="polars",
    context=ExecutionContext(
        mode=ExecutionMode.LAZY,
    ),
)

lazy_frame = lazy_result.output_handle.frame

print(type(lazy_frame))
lazy_frame


In [ ]:
lazy_frame.collect()


## 10. Expérimentations suivantes

Quelques axes utiles pour continuer à tester localement :

- ajouter des `cast()` ;
- tester `sort()` et `deduplicate()` ;
- tester les NULL dans les filtres ;
- comparer Pandas / Polars sur des datasets plus grands ;
- provoquer volontairement des erreurs de Schema ;
- inspecter les capabilities des adapters ;
- mesurer le coût du planning par rapport au coût physique.

Les fonctionnalités relationnelles multi-input (`join`, `union`, etc.) appartiennent au LOT-11 et ne sont pas encore disponibles dans cette version.
